In [1]:
import pandas as pd




splits = {'train': 'train.csv', 'validation': 'validation.csv', 'test': 'test.csv'}

    
df_train = pd.read_csv("hf://datasets/SahandNZ/cryptonews-articles-with-price-momentum-labels/" + splits["train"])
df_validation = pd.read_csv("hf://datasets/SahandNZ/cryptonews-articles-with-price-momentum-labels/" + splits["validation"])
df_test = pd.read_csv("hf://datasets/SahandNZ/cryptonews-articles-with-price-momentum-labels/" + splits["test"])


/Users/jamie/Downloads/LLM_Project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

dfs = [df_train, df_validation, df_test]
for df in dfs:
   
    df['date'] = pd.to_datetime(df['datetime']).dt.date  # for merging

In [3]:
df_train.head()

,datetime,text,url,label,date
0,2022-10-14,despite fact blockchainbased carbon credit mar...,https://cryptonews.com/news/bitcoin-price-and-...,1,2022-10-14
1,2022-10-14,trader gained huge kudos space predicting drop...,https://cryptonews.com/news/bitcoin-price-pred...,1,2022-10-14
2,2022-10-14,always worked sticking plan clear invalidation...,https://cryptonews.com/news/bitcoin-price-pred...,1,2022-10-14
3,2022-10-14,fact broke level system giving bullish signals...,https://cryptonews.com/news/bitcoin-price-pred...,1,2022-10-14
4,2022-10-14,demand coming confirms theres fuel keep going ...,https://cryptonews.com/news/bitcoin-price-pred...,1,2022-10-14


In [4]:
df_train.shape

(144276, 5)

In [5]:
import yfinance as yf

# Get the min and max dates from your news data
start_date = min(df_train['date'].min(), df_validation['date'].min(), df_test['date'].min())
end_date = max(df_train['date'].max(), df_validation['date'].max(), df_test['date'].max())
yf_end_date = end_date + pd.Timedelta(days=1)

btc = yf.download('BTC-USD', start=start_date, end=yf_end_date, interval='1d')
btc = btc.reset_index()  # 'Date' becomes a column
btc['date'] = btc['Date'].dt.date
btc.rename(columns={
    'Open': 'btc_open',
    'High': 'btc_high',
    'Low': 'btc_low',
    'Close': 'btc_close',
    'Volume': 'btc_volume'
}, inplace=True)
btc = btc[['date', 'btc_open', 'btc_high', 'btc_low', 'btc_close', 'btc_volume']]

[*********************100%***********************]  1 of 1 completed


In [6]:
btc

Price,date,btc_open,btc_high,btc_low,btc_close,btc_volume
Ticker,,BTC-USD,BTC-USD,BTC-USD,BTC-USD,BTC-USD
0,2022-10-14,19382.533203,19889.146484,19115.408203,19185.656250,38452356727
1,2022-10-15,19185.437500,19212.541016,19019.250000,19067.634766,16192235532
2,2022-10-16,19068.914062,19389.603516,19068.914062,19268.093750,17988916650
3,2022-10-17,19268.562500,19635.802734,19173.333984,19550.757812,27472552998
4,2022-10-18,19550.466797,19666.994141,19144.769531,19334.416016,30580012344
...,...,...,...,...,...,...
225,2023-05-27,26720.181641,26888.882812,26621.140625,26868.353516,7892015141
226,2023-05-28,26871.158203,28193.449219,26802.751953,28085.646484,14545229578
227,2023-05-29,28075.591797,28432.039062,27563.876953,27745.884766,15181308984


In [7]:
import requests
import json

def fetch_fng(start_ts, end_ts):
    url = f"https://api.alternative.me/fng/?limit=0&format=json&date_format=us"  # get all
    response = requests.get(url)
    data = response.json()['data']
    fng_list = []
    for item in data:
        fng_list.append({
            'date': pd.to_datetime(item['timestamp']).date(),
            'fng_value': int(item['value']),
            'fng_classification': item['value_classification']
        })
    df_fng = pd.DataFrame(fng_list)
    # Filter by your date range
    df_fng = df_fng[(df_fng['date'] >= start_date) & (df_fng['date'] <= end_date)]
    return df_fng

df_fng = fetch_fng(start_date, end_date)

In [8]:
df_fng

,date,fng_value,fng_classification
1014,2023-05-31,51,Neutral
1015,2023-05-30,51,Neutral
1016,2023-05-29,52,Neutral
1017,2023-05-28,50,Neutral
1018,2023-05-27,48,Neutral
...,...,...,...
1239,2022-10-18,22,Extreme Fear
1240,2022-10-17,20,Extreme Fear
1241,2022-10-16,24,Extreme Fear
1242,2022-10-15,24,Extreme Fear


In [9]:
df = pd.concat(
    [
        df_train.assign(split='train'),
        df_validation.assign(split='val'),
        df_test.assign(split='test')
    ],
    ignore_index=True
)

btc_for_merge = btc.copy()
if isinstance(btc_for_merge.columns, pd.MultiIndex):
    btc_for_merge.columns = [
        '_'.join([str(level) for level in col if level not in (None, '')]).strip('_')
        if isinstance(col, tuple) else str(col)
        for col in btc_for_merge.columns
    ]

def pick_col(df_columns, must_include):
    for col in df_columns:
        name = str(col).lower()
        if all(token in name for token in must_include):
            return col
    return None

date_col = pick_col(btc_for_merge.columns, ['date'])
open_col = pick_col(btc_for_merge.columns, ['open'])
high_col = pick_col(btc_for_merge.columns, ['high'])
low_col = pick_col(btc_for_merge.columns, ['low'])
close_col = pick_col(btc_for_merge.columns, ['close'])
volume_col = pick_col(btc_for_merge.columns, ['volume'])

btc_for_merge = btc_for_merge.rename(columns={
    date_col: 'date',
    open_col: 'btc_open',
    high_col: 'btc_high',
    low_col: 'btc_low',
    close_col: 'btc_close',
    volume_col: 'btc_volume'
})

btc_for_merge['date'] = pd.to_datetime(btc_for_merge['date']).dt.date
btc_for_merge = btc_for_merge[['date', 'btc_open', 'btc_high', 'btc_low', 'btc_close', 'btc_volume']]

df_merged = df.merge(btc_for_merge, on='date', how='left')
df_merged = df_merged.merge(df_fng, on='date', how='left')
df_merged.head()

,datetime,text,url,label,date,split,btc_open,btc_high,btc_low,btc_close,btc_volume,fng_value,fng_classification
0,2022-10-14,despite fact blockchainbased carbon credit mar...,https://cryptonews.com/news/bitcoin-price-and-...,1,2022-10-14,train,19382.533203,19889.146484,19115.408203,19185.65625,38452356727,24,Extreme Fear
1,2022-10-14,trader gained huge kudos space predicting drop...,https://cryptonews.com/news/bitcoin-price-pred...,1,2022-10-14,train,19382.533203,19889.146484,19115.408203,19185.65625,38452356727,24,Extreme Fear
2,2022-10-14,always worked sticking plan clear invalidation...,https://cryptonews.com/news/bitcoin-price-pred...,1,2022-10-14,train,19382.533203,19889.146484,19115.408203,19185.65625,38452356727,24,Extreme Fear
3,2022-10-14,fact broke level system giving bullish signals...,https://cryptonews.com/news/bitcoin-price-pred...,1,2022-10-14,train,19382.533203,19889.146484,19115.408203,19185.65625,38452356727,24,Extreme Fear
4,2022-10-14,demand coming confirms theres fuel keep going ...,https://cryptonews.com/news/bitcoin-price-pred...,1,2022-10-14,train,19382.533203,19889.146484,19115.408203,19185.65625,38452356727,24,Extreme Fear


In [10]:
df_train_after_merge = df_merged[df_merged['split'] == 'train']
df_validation_after_merge = df_merged[df_merged['split'] == 'val']
df_test_after_merge = df_merged[df_merged['split'] == 'test']

In [11]:
df_train_after_merge.head()

,datetime,text,url,label,date,split,btc_open,btc_high,btc_low,btc_close,btc_volume,fng_value,fng_classification
0,2022-10-14,despite fact blockchainbased carbon credit mar...,https://cryptonews.com/news/bitcoin-price-and-...,1,2022-10-14,train,19382.533203,19889.146484,19115.408203,19185.65625,38452356727,24,Extreme Fear
1,2022-10-14,trader gained huge kudos space predicting drop...,https://cryptonews.com/news/bitcoin-price-pred...,1,2022-10-14,train,19382.533203,19889.146484,19115.408203,19185.65625,38452356727,24,Extreme Fear
2,2022-10-14,always worked sticking plan clear invalidation...,https://cryptonews.com/news/bitcoin-price-pred...,1,2022-10-14,train,19382.533203,19889.146484,19115.408203,19185.65625,38452356727,24,Extreme Fear
3,2022-10-14,fact broke level system giving bullish signals...,https://cryptonews.com/news/bitcoin-price-pred...,1,2022-10-14,train,19382.533203,19889.146484,19115.408203,19185.65625,38452356727,24,Extreme Fear
4,2022-10-14,demand coming confirms theres fuel keep going ...,https://cryptonews.com/news/bitcoin-price-pred...,1,2022-10-14,train,19382.533203,19889.146484,19115.408203,19185.65625,38452356727,24,Extreme Fear


In [14]:
df_test_after_merge.head()

,datetime,text,url,label,date,split,btc_open,btc_high,btc_low,btc_close,btc_volume,fng_value,fng_classification
162310,2023-03-22,fact bitcoin moved different way equity market...,https://cryptonews.com/news/cathie-wood-of-ark...,0,2023-03-22,test,28158.720703,28803.335938,26759.996094,27307.4375,33382021890,62,Greed
162311,2023-03-22,corporate treasuries previously pulling away b...,https://cryptonews.com/news/cathie-wood-of-ark...,0,2023-03-22,test,28158.720703,28803.335938,26759.996094,27307.4375,33382021890,62,Greed
162312,2023-03-22,bitcoin 27 past two weeks according data coing...,https://cryptonews.com/news/cathie-wood-of-ark...,0,2023-03-22,test,28158.720703,28803.335938,26759.996094,27307.4375,33382021890,62,Greed
162313,2023-03-22,note investors ark invest argued rally sign bi...,https://cryptonews.com/news/cathie-wood-of-ark...,0,2023-03-22,test,28158.720703,28803.335938,26759.996094,27307.4375,33382021890,62,Greed
162314,2023-03-22,fed left quantitative tightening schedule allo...,https://cryptonews.com/news/long-liquidations-...,0,2023-03-22,test,28158.720703,28803.335938,26759.996094,27307.4375,33382021890,62,Greed


In [13]:
df_train_after_merge.to_parquet("train_after_merge.parquet", index=False)
df_validation_after_merge.to_parquet("validation_after_merge.parquet", index=False)
df_test_after_merge.to_parquet("test_after_merge.parquet", index=False)